## GPT 3.5-Turbo DSPy 

In [2]:
import dspy
import pandas as pd
import os
import json

# Set up a basic teleprompter forcompiling the program.
from dspy.teleprompt import BootstrapFewShot

# Set up the `evaluation` function. 
from dspy.evaluate.evaluate import Evaluate

#### Load in Training Examples

In [28]:
# import data from 20 manual examples as training and 50 gpt examples as validation
train_examples=pd.read_csv('~/20 Manual Labeling.csv')
print(train_examples.shape)
train_examples.head(3)

dev_examples=pd.read_csv('~/50examples_for_DSPy_withJson.csv')
print(dev_examples.shape)
dev_examples.head(3)

(20, 16)
(46, 16)


,id,body,min_edulevels_name,employment_type_name,min_years_experience,remote_type_name,salary_to,city_name,state_name,title_name,specialized_skills_name,certifications_name,common_skills_name,naics2_name,onet_name,pred_json
0,18ec1d38a8acc34f...,"Derrickhand, Buc...",High school or GED,Full-time (> 32 ...,1.0,[None],NaN,"Buckhannon, WV",West Virginia,Derrickhands,['Well Services'...,['CDL Class B Li...,['Customer Servi...,Unclassified Ind...,"Roustabouts, Oil...","{\n ""position..."
1,0b8c421fca565f14...,Business Relatio...,High school or GED,Full-time (> 32 ...,4.0,[None],45000.0,"Mount Airy, NC",North Carolina,Business Relatio...,['Public Adminis...,[],['Presentations'...,Unclassified Ind...,Rehabilitation C...,"{\n ""position..."
2,256704ef9f689ef8...,Pharmacy Technic...,High school or GED,Full-time (> 32 ...,NaN,[None],39520.0,"San Quentin, CA",California,Pharmacy Technic...,['Medical Prescr...,['Certified Phar...,['Customer Servi...,Unclassified Ind...,Pharmacy Technic...,"{\n ""position..."


#### Format Training Examples to DSPy format

In [35]:
# set up training dataset and validation dataset
train_results = list(train_examples['pred_json'])
train_contents=list(train_examples['body'])
print(len(train_results),len(train_contents))

dev_results = list(dev_examples['pred_json'])
dev_contents=list(dev_examples['body'])
print(len(dev_results),len(dev_contents))

train_example_list=[dspy.Example(context=content, answer=result) for content, result in zip(train_contents, train_results)]
dev_example_list=[dspy.Example(context=content, answer=result) for content, result in zip(dev_contents, dev_results)]


trainset=train_example_list
devset=dev_example_list[:5] # just use first 5 examples for testing for now

# 'body' field is the input. 'pred_json'are labels and/or metadata.
trainset = [x.with_inputs('context') for x in trainset]
devset = [x.with_inputs('context') for x in devset]

print(len(trainset),len(devset))

20 20
46 46
20 5


#### Connect to Open AI API Key and create model Signature and Module

In [36]:
# Connect OpenAI gpt3.5 turbo to DSPy
api_key=os.getenv('openai_key')
turbo = dspy.OpenAI(model='gpt-3.5-turbo', api_key=api_key)
dspy.settings.configure(lm=turbo)


# create signature for input and output
class GenerateAnswer(dspy.Signature):
    """Answer questions with short factoid answers."""

    context = dspy.InputField(desc="contain relevant facts")
    question = dspy.InputField(desc="unique possible questions")
    answer = dspy.OutputField(
    desc="Answers should be a JSON object containing key-value pairs, with the keys being: 'position_title', 'location', 'work_arrangement', 'experience', 'employment_type', 'pay', 'degree_certification', and 'required_skills'. The values should be between 1 and 30 words. If you do not know the answer, just state 'Not Specified'"
)

# Create module using dspy module
class QUESTIONANSWER(dspy.Module):
    def __init__(self,question):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer, max_tokens=400)
        self.question=question

    def forward(self, context):
        question=self.question
        pred = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context,answer=pred.answer)


uncompiled=QUESTIONANSWER('''
                             1. What is the title of this position?
                             2. Where is this position located, including city, state and zip code?
                             3. What is the work arrangement for this position, remote, hybrid, or on-site?
                             4. what are years of experience required for this position?
                             5. What is the employment type, full time, part time, or internship?
                             6. What is the pay for this position?
                             7. What is required degree or certification?
                             8. What are required skills?
                              ''')


#### Validate Uncompiled vs Compiled

In [38]:
# Validation logic: check that the predicted answer is correct using 'exact_match'
    # we perhaps need to create our own metric or use another LLM as judge
def validate_answer(example, pred, trace=None):
    answer_EM = dspy.evaluate.answer_exact_match(example, pred)
    return answer_EM

teleprompter = BootstrapFewShot(metric=validate_answer)

# Compile
compiled = teleprompter.compile(uncompiled, trainset=trainset)

evaluation = Evaluate(devset=devset, num_threads=1, display_progress=True, display_table=5)

# below code will check the accuracy code before optimizing
prev_score=evaluation(uncompiled, metric=validate_answer)

# get the accuracy after optimizing
improved_score=evaluation(compiled, metric=validate_answer)

100%|██████████| 20/20 [00:00<00:00, 748.72it/s]


Average Metric: 0 / 5  (0.0): 100%|██████████| 5/5 [00:00<00:00, 1684.59it/s]


,example_context,example_answer,pred_context,pred_answer,validate_answer
0,"Derrickhand, Buckhannon, WV Job Order Number WV2925359 Post Date 05/12/2023 Job Location Buckhannon, West Virginia 26201 County Upshur Job Summary Job Summary: This position is...","{ ""position_title"": ""Derrickhand"", ""location"": ""Buckhannon, West Virginia 26201"", ""work_arrange"": ""On-site"", ""experience"": ""1-2 years of Workover - Derrickhand experience required"", ""employment_type"": ""Full-time"", ""pay"": ""$0/hour"", ""degree_certification"": ""High...","Derrickhand, Buckhannon, WV Job Order Number WV2925359 Post Date 05/12/2023 Job Location Buckhannon, West Virginia 26201 County Upshur Job Summary Job Summary: This position is...","{ ""position_title"": ""Derrickhand"", ""location"": ""Buckhannon, West Virginia 26201"", ""work_arrangement"": ""On-site"", ""experience"": ""1-2 years of Workover - Derrickhand experience required"", ""employment_type"": ""Full time"", ""pay"": ""$0 /...",False
1,"Business Relations Representative Mount Airy, NC $32,703 - $51,904 a year - Full-time Job details Salary $32,703 - $51,904 a year Job Type Full-time Full...","{ ""position_title"": ""Business Relations Representative"", ""location"": ""Mount Airy, NC 27030"", ""work_arrange"": ""On-site"", ""experience"": ""Previous job placement experience preferred"", ""employment_type"": ""Full-time"", ""pay"": ""$32,703 - $51,904 a...","Business Relations Representative Mount Airy, NC $32,703 - $51,904 a year - Full-time Job details Salary $32,703 - $51,904 a year Job Type Full-time Full...","{ ""position_title"": ""Business Relations Representative"", ""location"": ""Mount Airy, NC 27030"", ""work_arrangement"": ""On-site"", ""experience"": ""Not Specified"", ""employment_type"": ""Full-time"", ""pay"": ""$32,703 - $51,904 a year"", ""degree_certification"": ""Bachelor's...",False
2,"Pharmacy Technician San Quentin, CA $18 - $19 an hour - Contract Profile insights Find out how your skills align with the job description Licenses...","{ ""position_title"": ""Pharmacy Technician"", ""location"": ""San Quentin, CA 94964"", ""work_arrange"": ""On-site"", ""experience"": ""1 year"", ""employment_type"": ""Contract"", ""pay"": ""$18 - $19 an hour"", ""degree_certification"": ""Pharmacy Technician...","Pharmacy Technician San Quentin, CA $18 - $19 an hour - Contract Profile insights Find out how your skills align with the job description Licenses...","{ ""position_title"": ""Pharmacy Technician"", ""location"": ""San Quentin, CA 94964"", ""work_arrangement"": ""On-site"", ""experience"": ""1 year"", ""employment_type"": ""Contract"", ""pay"": ""$18.00 - $19.00 per hour"", ""degree_certification"": ""Pharmacy Technician...",False
3,"Marketing Strategist Maryville, TN Full-time Profile insights Find out how your skills align with the job description Skills Do you have experience in SharePoint ?...","{ ""position_title"": ""Retail Social Marketing Strategist"", ""location"": ""Maryville, TN"", ""work_arrange"": ""On-site"", ""experience"": ""4+ years of marketing experience preferred"", ""employment_type"": ""Full-time"", ""pay"": ""Competitive"", ""degree_certification"": ""Bachelor's Degree...","Marketing Strategist Maryville, TN Full-time Profile insights Find out how your skills align with the job description Skills Do you have experience in SharePoint ?...","{ ""position_title"": ""Retail Social Marketing Strategist"", ""location"": ""Maryville, TN"", ""work_arrangement"": ""On-site"", ""experience"": ""4+ years of marketing experience preferred"", ""employment_type"": ""Full-time"", ""pay"": ""Not Specified"", ""degree_certification"": ""Bachelor's...",False
4,"GIS Technician Dobson Fiber 14101 Wireless Way, Oklahoma City, OK 73134 Full-time Dobson Fiber 16 reviews Read what people are saying about working here. Job...","{ ""position_title"": ""GIS Technician"", ""location"": ""Oklahoma City, OK 73134"", ""work_arrange"": ""On-site"", ""experience"": ""3-5 years

Average Metric: 0 / 5  (0.0): 100%|██████████| 5/5 [00:00<00:00, 669.89it/s]


,example_context,example_answer,pred_context,pred_answer,validate_answer
0,"Derrickhand, Buckhannon, WV Job Order Number WV2925359 Post Date 05/12/2023 Job Location Buckhannon, West Virginia 26201 County Upshur Job Summary Job Summary: This position is...","{ ""position_title"": ""Derrickhand"", ""location"": ""Buckhannon, West Virginia 26201"", ""work_arrange"": ""On-site"", ""experience"": ""1-2 years of Workover - Derrickhand experience required"", ""employment_type"": ""Full-time"", ""pay"": ""$0/hour"", ""degree_certification"": ""High...","Derrickhand, Buckhannon, WV Job Order Number WV2925359 Post Date 05/12/2023 Job Location Buckhannon, West Virginia 26201 County Upshur Job Summary Job Summary: This position is...","{ ""position_title"": ""Derrickhand"", ""location"": ""Buckhannon, West Virginia 26201"", ""work_arrangement"": ""On-site"", ""experience"": ""1-2 years of Workover - Derrickhand experience required"", ""employment_type"": ""Rotary Drill Operators, Oil and...",False
1,"Business Relations Representative Mount Airy, NC $32,703 - $51,904 a year - Full-time Job details Salary $32,703 - $51,904 a year Job Type Full-time Full...","{ ""position_title"": ""Business Relations Representative"", ""location"": ""Mount Airy, NC 27030"", ""work_arrange"": ""On-site"", ""experience"": ""Previous job placement experience preferred"", ""employment_type"": ""Full-time"", ""pay"": ""$32,703 - $51,904 a...","Business Relations Representative Mount Airy, NC $32,703 - $51,904 a year - Full-time Job details Salary $32,703 - $51,904 a year Job Type Full-time Full...","{ ""position_title"": ""Business Relations Representative"", ""location"": ""Mount Airy, NC 27030"", ""work_arrangement"": ""On-site"", ""experience"": ""Not Specified"", ""employment_type"": ""Full-time"", ""pay"": ""$32,703 - $51,904 a year"", ""degree_certification"": ""Bachelor's...",False
2,"Pharmacy Technician San Quentin, CA $18 - $19 an hour - Contract Profile insights Find out how your skills align with the job description Licenses...","{ ""position_title"": ""Pharmacy Technician"", ""location"": ""San Quentin, CA 94964"", ""work_arrange"": ""On-site"", ""experience"": ""1 year"", ""employment_type"": ""Contract"", ""pay"": ""$18 - $19 an hour"", ""degree_certification"": ""Pharmacy Technician...","Pharmacy Technician San Quentin, CA $18 - $19 an hour - Contract Profile insights Find out how your skills align with the job description Licenses...","{ ""position_title"": ""Pharmacy Technician"", ""location"": ""San Quentin, CA, 94964"", ""work_arrangement"": ""On-site"", ""experience"": ""1 year of experience as Pharmacy Technician"", ""employment_type"": ""Contract"", ""pay"": ""$18.00 - $19.00...",False
3,"Marketing Strategist Maryville, TN Full-time Profile insights Find out how your skills align with the job description Skills Do you have experience in SharePoint ?...","{ ""position_title"": ""Retail Social Marketing Strategist"", ""location"": ""Maryville, TN"", ""work_arrange"": ""On-site"", ""experience"": ""4+ years of marketing experience preferred"", ""employment_type"": ""Full-time"", ""pay"": ""Competitive"", ""degree_certification"": ""Bachelor's Degree...","Marketing Strategist Maryville, TN Full-time Profile insights Find out how your skills align with the job description Skills Do you have experience in SharePoint ?...","{ ""position_title"": ""Retail Social Marketing Strategist"", ""location"": ""Maryville, TN"", ""work_arrangement"": ""On-site"", ""experience"": ""4+ years of marketing experience preferred"", ""employment_type"": ""Full-time"", ""pay"": ""Not Specified"", ""degree_certification"": ""Bachelor's...",False
4,"GIS Technician Dobson Fiber 14101 Wireless Way, Oklahoma City, OK 73134 Full-time Dobson Fiber 16 reviews Read what people are saying about working here. Job...","{ ""position_title"": ""GIS Technician"", ""location"": ""Oklahoma City, OK 73134"", ""work_arrange"": ""On-site"", ""experience"": ""3-5 years GIS experience"", 

#### Create GPT labeled examples after compiling

In [39]:
def create_json_append_to_df(job_posting):
    # Generate answer from DSPy
    pred = compiled(context=job_posting) # before, I used uncomplied(), now I changed it to compiled(). All the rest stays the same
    posting = pred.answer

    # Extract the JSON object from the posting
    start_index = posting.find('{')
    end_index = posting.find('}', start_index) + 1

    if start_index == -1 or end_index == -1:
        print("No valid JSON object found in the posting")
        return None

    json_string = posting[start_index:end_index]

    try:
        job_details = json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return None

    # print(job_details)

    pred_json = json.dumps(job_details, indent=4)
    print('Done, Output Succeed')
    # print(pred_json)

    return pred_json

examples_json=dev_examples.copy()
examples_json['pred_json_after_compiling']=examples_json['body'].apply(lambda x: create_json_append_to_df(x))
print(examples_json.shape)
display(examples_json.tail(5))

Done, Output Succeed
Done, Output Succeed
Done, Output Succeed
Done, Output Succeed
Done, Output Succeed
Done, Output Succeed
Done, Output Succeed


INFO:backoff:Backing off request(...) for 0.2s (openai.error.RateLimitError: Rate limit reached for gpt-3.5-turbo in organization org-LnLUOX2GedQp7zTstR5xBRKj on tokens per min (TPM): Limit 60000, Used 44873, Requested 17570. Please try again in 2.443s. Visit https://platform.openai.com/account/rate-limits to learn more.)


Backing off 0.2 seconds after 1 tries calling function <function GPT3.request at 0x10c162290> with kwargs {'max_tokens': 200, 'n': 1, 'temperature': 0.0}


INFO:backoff:Backing off request(...) for 1.4s (openai.error.RateLimitError: Rate limit reached for gpt-3.5-turbo in organization org-LnLUOX2GedQp7zTstR5xBRKj on tokens per min (TPM): Limit 60000, Used 44489, Requested 17570. Please try again in 2.059s. Visit https://platform.openai.com/account/rate-limits to learn more.)


Backing off 1.4 seconds after 2 tries calling function <function GPT3.request at 0x10c162290> with kwargs {'max_tokens': 200, 'n': 1, 'temperature': 0.0}


...

INFO:backoff:Backing off request(...) for 0.2s (openai.error.RateLimitError: Rate limit reached for gpt-3.5-turbo in organization org-LnLUOX2GedQp7zTstR5xBRKj on tokens per min (TPM): Limit 60000, Used 46192, Requested 17358. Please try again in 3.55s. Visit https://platform.openai.com/account/rate-limits to learn more.)


Backing off 0.2 seconds after 4 tries calling function <function GPT3.request at 0x10c162290> with kwargs {'max_tokens': 200, 'n': 1, 'temperature': 0.0}


INFO:backoff:Backing off request(...) for 15.2s (openai.error.RateLimitError: Rate limit reached for gpt-3.5-turbo in organization org-LnLUOX2GedQp7zTstR5xBRKj on tokens per min (TPM): Limit 60000, Used 45909, Requested 17358. Please try again in 3.267s. Visit https://platform.openai.com/account/rate-limits to learn more.)


Backing off 15.2 seconds after 5 tries calling function <function GPT3.request at 0x10c162290> with kwargs {'max_tokens': 200, 'n': 1, 'temperature': 0.0}
Done, Output Succeed
(46, 17)


,id,body,min_edulevels_name,employment_type_name,min_years_experience,remote_type_name,salary_to,city_name,state_name,title_name,specialized_skills_name,certifications_name,common_skills_name,naics2_name,onet_name,pred_json,pred_json_after_compiling
41,1f9b129547a8bc90...,Logistics Coordi...,High school or GED,Full-time (> 32 ...,NaN,[None],29120.0,"Henderson, NV",Nevada,Logistics Coordi...,['Continuous Imp...,[],"['Management', '...",Unclassified Ind...,Logisticians,"{\n ""position...","{\n ""position..."
42,a12030480f74113b...,Systems Administ...,Bachelor's degree,Full-time (> 32 ...,3.0,[None],NaN,"Laurel, MD",Maryland,Systems and Secu...,['Traceability M...,['CompTIA Securi...,"['Research', 'In...",Unclassified Ind...,Network and Comp...,"{\n ""position...","{\n ""position..."
43,d5cd5a4da6da14dc...,"Between $52,250 ...",Bachelor's degree,Full-time (> 32 ...,6.0,[None],78390.0,"Jefferson City, MO",Missouri,Technical Writers,"['Copy Editing',...",[],['Customer Servi...,Unclassified Ind...,Technical Writers,"{\n ""position...","{\n ""position..."
44,b7e2dc5d319e8a5a...,Sr. Net Develope...,Bachelor's degree,Full-time (> 32 ...,2.0,[None],NaN,"Irving, TX",Texas,.NET Developers,"['Code Review', ...",[],['Operations'],Unclassified Ind...,Software Developers,"{\n ""position...","{\n ""position..."
45,9c61f7202efc3e92...,Federal Sales En...,Bachelor's degree,Full-time (> 32 ...,NaN,Remote,120000.0,"Walpole, MA",Massachusetts,Federal Sales En...,['Sales Forecast...,['Product Certif...,['Sales'],Unclassified Ind...,Sales Engineers,"{\n ""position...","{\n ""position..."


In [43]:
examples_json_with_manual_labels=examples_json.copy()
merged_df = pd.merge(examples_json,train_examples[['id', 'pred_json']], on='id', how='left')
merged_df.rename(columns={'pred_json': 'manual_labeling'}, inplace=True)

merged_df.head(3)

,id,body,min_edulevels_name,employment_type_name,min_years_experience,remote_type_name,salary_to,city_name,state_name,title_name,specialized_skills_name,certifications_name,common_skills_name,naics2_name,onet_name,pred_json_x,pred_json_after_compiling,pred_json_y
0,18ec1d38a8acc34f...,"Derrickhand, Buc...",High school or GED,Full-time (> 32 ...,1.0,[None],NaN,"Buckhannon, WV",West Virginia,Derrickhands,['Well Services'...,['CDL Class B Li...,['Customer Servi...,Unclassified Ind...,"Roustabouts, Oil...","{\n ""position...","{\n ""position...","{\n ""position..."
1,0b8c421fca565f14...,Business Relatio...,High school or GED,Full-time (> 32 ...,4.0,[None],45000.0,"Mount Airy, NC",North Carolina,Business Relatio...,['Public Adminis...,[],['Presentations'...,Unclassified Ind...,Rehabilitation C...,"{\n ""position...","{\n ""position...",NaN
2,256704ef9f689ef8...,Pharmacy Technic...,High school or GED,Full-time (> 32 ...,NaN,[None],39520.0,"San Quentin, CA",California,Pharmacy Technic...,['Medical Prescr...,['Certified Phar...,['Customer Servi...,Unclassified Ind...,Pharmacy Technic...,"{\n ""position...","{\n ""position...","{\n ""position..."
